# Molecule Analysis Data Audit

This notebook profiles the datasets in `data/` and produces a compact inventory for planning math, machine learning, and graph-based work.

Goals:
- inventory every dataset file
- classify likely task type
- measure missingness and identifier quality
- surface first-wave datasets for implementation

In [ ]:
from __future__ import annotations

import csv
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd


PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
assert DATA_DIR.exists(), f"Expected data directory at {DATA_DIR}"

## Audit Helpers

In [ ]:
@dataclass
class DatasetSummary:
    file_name: str
    suffix: str
    size_kb: float
    row_count: int | None
    column_count: int | None
    task_type: str
    target_columns: list[str]
    candidate_keys: list[str]
    best_key: str | None
    best_key_unique: bool | None
    missing_cells: int | None
    missing_rate: float | None
    notes: str


def infer_task_type(file_name: str, columns: list[str]) -> str:
    lower_name = file_name.lower()
    lower_cols = [col.lower() for col in columns]

    if lower_name.endswith('.mat'):
        return 'quantum-regression-mat'
    if lower_name in {'qm8.csv', 'qm9.csv'}:
        return 'multi-target-regression'
    if lower_name in {'delaney-processed.csv', 'lipophilicity.csv', 'sampl.csv', 'bace.csv'}:
        return 'regression'
    if lower_name in {'bbbp.csv', 'clintox.csv', 'hiv.csv'}:
        return 'binary-classification'
    if lower_name in {'tox21.csv', 'sider.csv', 'muv.csv', 'toxcast_data.csv'}:
        return 'multitask-classification'
    binary_like = [col for col in lower_cols if col not in {'smiles', 'mol_id', 'cid', 'cmpd_chemblid', 'compound id', 'iupac', 'name'}]
    if len(binary_like) > 3:
        return 'multitask-or-multilabel'
    return 'unknown'


def infer_target_columns(file_name: str, columns: list[str]) -> list[str]:
    lower_name = file_name.lower()
    if lower_name == 'bbbp.csv':
        return ['p_np']
    if lower_name == 'clintox.csv':
        return ['FDA_APPROVED', 'CT_TOX']
    if lower_name == 'hiv.csv':
        return ['activity', 'HIV_active']
    if lower_name == 'lipophilicity.csv':
        return ['exp']
    if lower_name == 'sampl.csv':
        return ['expt', 'calc']
    if lower_name == 'delaney-processed.csv':
        return ['measured log solubility in mols per litre']
    if lower_name == 'bace.csv':
        return ['Class', 'pIC50']
    if lower_name == 'tox21.csv':
        return columns[:12]
    if lower_name == 'sider.csv':
        return columns[1:]
    if lower_name == 'muv.csv':
        return columns[:-2]
    if lower_name == 'toxcast_data.csv':
        return [col for col in columns if col != 'smiles']
    if lower_name == 'qm8.csv':
        return [col for col in columns if col != 'smiles']
    if lower_name == 'qm9.csv':
        return [col for col in columns if col not in {'mol_id', 'smiles'}]
    return []


def candidate_key_columns(columns: list[str]) -> list[str]:
    priority_terms = ('id', 'smiles', 'name', 'iupac', 'cid', 'chembl')
    matches = []
    for col in columns:
        low = col.lower()
        if any(term in low for term in priority_terms):
            matches.append(col)
    return matches


def key_profile(rows: list[dict[str, str]], columns: list[str]) -> tuple[list[str], str | None, bool | None]:
    candidates = candidate_key_columns(columns)
    best_key = None
    best_key_unique = None

    for col in candidates:
        values = [row[col].strip() for row in rows]
        nonempty = [value for value in values if value != '']
        if len(nonempty) != len(rows):
            continue
        is_unique = len(set(nonempty)) == len(nonempty)
        if is_unique:
            best_key = col
            best_key_unique = True
            if col.lower() in {'mol_id', 'cid', 'cmpd_chemblid', 'compound id', 'num', 'smiles'}:
                break
        if best_key is None:
            best_key = col
            best_key_unique = False

    return candidates, best_key, best_key_unique


def summarize_csv(path: Path) -> DatasetSummary:
    with path.open(newline='') as handle:
        rows = list(csv.DictReader(handle))
        columns = rows[0].keys() if rows else []

    column_list = list(columns)
    total_cells = len(rows) * len(column_list) if rows else 0
    missing_cells = sum(1 for row in rows for value in row.values() if value.strip() == '')
    missing_rate = (missing_cells / total_cells) if total_cells else None
    candidates, best_key, best_key_unique = key_profile(rows, column_list)

    notes = []
    duplicate_headers = [col for col, count in Counter(column_list).items() if count > 1]
    if duplicate_headers:
        notes.append(f'duplicate headers: {duplicate_headers}')
    if path.name == 'toxcast_data.csv':
        notes.append('wide sparse multitask assay matrix')
    if path.name == 'qm8.csv':
        notes.append('duplicate PBE0 headers need normalization before modeling')
    if path.name == 'qm9.csv':
        notes.append('multi-target quantum regression with clean mol_id')

    return DatasetSummary(
        file_name=path.name,
        suffix=path.suffix,
        size_kb=round(path.stat().st_size / 1024, 2),
        row_count=len(rows),
        column_count=len(column_list),
        task_type=infer_task_type(path.name, column_list),
        target_columns=infer_target_columns(path.name, column_list),
        candidate_keys=candidates,
        best_key=best_key,
        best_key_unique=best_key_unique,
        missing_cells=missing_cells,
        missing_rate=missing_rate,
        notes='; '.join(notes),
    )


def summarize_mat(path: Path) -> DatasetSummary:
    return DatasetSummary(
        file_name=path.name,
        suffix=path.suffix,
        size_kb=round(path.stat().st_size / 1024, 2),
        row_count=None,
        column_count=None,
        task_type='quantum-regression-mat',
        target_columns=[],
        candidate_keys=[],
        best_key=None,
        best_key_unique=None,
        missing_cells=None,
        missing_rate=None,
        notes='MATLAB binary file; inspect later with scipy.io.loadmat or h5py if needed',
    )

In [ ]:
summaries: list[DatasetSummary] = []
for path in sorted(DATA_DIR.iterdir()):
    if path.suffix.lower() == '.csv':
        summaries.append(summarize_csv(path))
    elif path.suffix.lower() == '.mat':
        summaries.append(summarize_mat(path))

summary_df = pd.DataFrame([summary.__dict__ for summary in summaries])
summary_df['missing_rate'] = summary_df['missing_rate'].round(4)
summary_df = summary_df.sort_values(['task_type', 'file_name']).reset_index(drop=True)
display(summary_df)

## First-Wave Recommendations

In [ ]:
tier_a = {'BBBP.csv', 'clintox.csv', 'delaney-processed.csv', 'Lipophilicity.csv', 'SAMPL.csv'}
tier_b = {'HIV.csv', 'tox21.csv', 'sider.csv', 'muv.csv'}
tier_c = {'toxcast_data.csv', 'qm8.csv', 'qm9.csv', 'qm7b.mat', 'bace.csv'}

def assign_tier(file_name: str) -> str:
    if file_name in tier_a:
        return 'A'
    if file_name in tier_b:
        return 'B'
    if file_name in tier_c:
        return 'C'
    return 'review'

summary_df['tier'] = summary_df['file_name'].map(assign_tier)
display(summary_df[['file_name', 'tier', 'task_type', 'row_count', 'column_count', 'best_key', 'best_key_unique', 'missing_rate', 'notes']])

In [ ]:
plot_df = summary_df.dropna(subset=['row_count', 'column_count']).copy()
plot_df['shape_label'] = plot_df['row_count'].astype(int).astype(str) + ' x ' + plot_df['column_count'].astype(int).astype(str)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=plot_df.sort_values('row_count', ascending=False), x='row_count', y='file_name', hue='tier', ax=axes[0])
axes[0].set_title('Dataset Row Counts')
axes[0].set_xlabel('Rows')
axes[0].set_ylabel('Dataset')

sns.barplot(data=plot_df.sort_values('missing_rate', ascending=False), x='missing_rate', y='file_name', hue='tier', ax=axes[1])
axes[1].set_title('Missingness Rate')
axes[1].set_xlabel('Missing fraction')
axes[1].set_ylabel('Dataset')

plt.tight_layout()
plt.show()

## Deep Dive: Identifier Quality

In [ ]:
def inspect_identifier_quality(path: Path) -> pd.DataFrame:
    rows = list(csv.DictReader(path.open(newline='')))
    columns = list(rows[0].keys())
    candidates = candidate_key_columns(columns)
    records = []
    for col in candidates:
        values = [row[col].strip() for row in rows]
        nonempty = [v for v in values if v]
        unique_count = len(set(nonempty))
        records.append({
            'column': col,
            'nonempty': len(nonempty),
            'unique': unique_count,
            'duplicate_values': sum(1 for _, count in Counter(nonempty).items() if count > 1),
            'is_complete': len(nonempty) == len(rows),
            'is_unique': unique_count == len(nonempty),
        })
    return pd.DataFrame(records).sort_values(['is_unique', 'is_complete', 'column'], ascending=[False, False, True])

for file_name in ['BBBP.csv', 'clintox.csv', 'delaney-processed.csv', 'Lipophilicity.csv', 'SAMPL.csv', 'tox21.csv', 'toxcast_data.csv', 'qm8.csv', 'qm9.csv']:
    path = DATA_DIR / file_name
    print(f'\n### {file_name}')
    display(inspect_identifier_quality(path))

## Actionable Output

Use this table as the contract for the next notebooks: which dataset to start with, which identifier to trust, and whether target columns need missing-label handling.

In [ ]:
starter_plan = summary_df.loc[summary_df['tier'] == 'A', [
    'file_name', 'task_type', 'target_columns', 'best_key', 'best_key_unique', 'missing_rate', 'notes'
]].copy()
starter_plan['recommended_role'] = [
    'binary classification baseline' if task == 'binary-classification' else 'regression baseline'
    for task in starter_plan['task_type']
]
display(starter_plan.sort_values(['task_type', 'file_name']))